In [5]:
def read_board(filename):
    board = []
    with open(filename, 'r') as file:
        for line in file:
            row = []
            line = line.strip()
            for ch in line:
                row.append(int(ch))
            board.append(row)
    return board

def print_board(board):
    for i in range(9):
        if i % 3 == 0 and i != 0:
            print("-" * 21)
        for j in range(9):
            if j % 3 == 0 and j != 0:
                print("|", end=" ")
            print(board[i][j], end=" ")
        print()

def get_subgrid_index(row, col):
    return (row // 3) * 3 + (col // 3)

def get_possible_values(board, row, col, domains):
    if board[row][col] != 0:
        return [board[row][col]]
    
    possible = []
    for value in range(1, 10):
        conflict = False
        
        for j in range(9):
            if board[row][j] == value:
                conflict = True
                break
        
        if not conflict:
            for i in range(9):
                if board[i][col] == value:
                    conflict = True
                    break
        
        if not conflict:
            start_row = (row // 3) * 3
            start_col = (col // 3) * 3
            for i in range(3):
                for j in range(3):
                    if board[start_row + i][start_col + j] == value:
                        conflict = True
                        break
                if conflict:
                    break
        
        if not conflict:
            possible.append(value)
    
    return possible

def initialize_domains(board):
    domains = {}
    for i in range(9):
        for j in range(9):
            if board[i][j] == 0:
                domains[(i, j)] = get_possible_values(board, i, j, domains)
            else:
                domains[(i, j)] = [board[i][j]]
    return domains

def get_neighbors(row, col):
    neighbors = set()
    
    for j in range(9):
        if j != col:
            neighbors.add((row, j))
    
    for i in range(9):
        if i != row:
            neighbors.add((i, col))
    
    start_row = (row // 3) * 3
    start_col = (col // 3) * 3
    for i in range(3):
        for j in range(3):
            if start_row + i != row and start_col + j != col:
                neighbors.add((start_row + i, start_col + j))
    
    return list(neighbors)

def ac3(domains):
    queue = []
    for i in range(9):
        for j in range(9):
            neighbors = get_neighbors(i, j)
            for neighbor in neighbors:
                queue.append(((i, j), neighbor))
    
    while queue:
        var1, var2 = queue.pop(0)
        if revise(domains, var1, var2):
            if len(domains[var1]) == 0:
                return False
            neighbors = get_neighbors(var1[0], var1[1])
            for neighbor in neighbors:
                if neighbor != var2:
                    queue.append((neighbor, var1))
    return True

def revise(domains, var1, var2):
    revised = False
    to_remove = []
    
    for value in domains[var1]:
        possible = False
        for neighbor_value in domains[var2]:
            if value != neighbor_value:
                possible = True
                break
        if not possible:
            to_remove.append(value)
            revised = True
    
    for value in to_remove:
        domains[var1].remove(value)
    
    return revised

def select_unassigned_variable(board, domains):
    min_remaining = 10
    best_cell = None
    
    for i in range(9):
        for j in range(9):
            if board[i][j] == 0:
                num_possible = len(domains[(i, j)])
                if num_possible < min_remaining:
                    min_remaining = num_possible
                    best_cell = (i, j)
                    if min_remaining == 1:
                        return best_cell
    
    return best_cell

def forward_check(board, domains, row, col, value):
    new_domains = {}
    for key in domains:
        new_domains[key] = domains[key][:]
    
    new_domains[(row, col)] = [value]
    
    neighbors = get_neighbors(row, col)
    for neighbor in neighbors:
        if board[neighbor[0]][neighbor[1]] == 0:
            if value in new_domains[neighbor]:
                new_domains[neighbor].remove(value)
                if len(new_domains[neighbor]) == 0:
                    return None
    
    return new_domains

backtrack_calls = 0
backtrack_failures = 0

def backtrack(board, domains):
    global backtrack_calls, backtrack_failures
    backtrack_calls = backtrack_calls + 1
    
    for i in range(9):
        for j in range(9):
            if board[i][j] == 0:
                row, col = i, j
                possible_values = domains[(row, col)]
                
                for value in possible_values:
                    board[row][col] = value
                    
                    new_domains = forward_check(board, domains, row, col, value)
                    
                    if new_domains is not None:
                        ac3_success = ac3(new_domains)
                        
                        if ac3_success:
                            result = backtrack(board, new_domains)
                            if result is not None:
                                return result
                    
                    board[row][col] = 0
                
                backtrack_failures = backtrack_failures + 1
                return None
    
    return board

def solve_sudoku(filename):
    global backtrack_calls, backtrack_failures
    backtrack_calls = 0
    backtrack_failures = 0
    
    print("Solving:", filename)
    print("Initial board:")
    board = read_board(filename)
    print_board(board)
    print()
    
    domains = initialize_domains(board)
    
    ac3(domains)
    
    solution = backtrack(board, domains)
    
    if solution is not None:
        print("Solution found:")
        print_board(solution)
        print()
        print("Backtrack function calls:", backtrack_calls)
        print("Backtrack failures:", backtrack_failures)
        print("Success rate: {:.1f}%".format((backtrack_calls - backtrack_failures) / backtrack_calls * 100))
        print()
        print("=" * 50)
        print()
        return solution
    else:
        print("No solution found")
        return None

solve_sudoku("easy.txt")
solve_sudoku("medium.txt")
solve_sudoku("hard.txt")
solve_sudoku("veryhard.txt")

Solving: easy.txt
Initial board:
0 0 4 | 0 3 0 | 0 5 0 
6 0 9 | 4 0 0 | 0 0 0 
0 0 5 | 1 0 0 | 4 8 9 
---------------------
0 0 0 | 0 6 0 | 9 3 0 
3 0 0 | 8 0 7 | 0 0 2 
0 2 6 | 0 4 0 | 0 0 0 
---------------------
4 5 3 | 0 0 9 | 6 0 0 
0 0 0 | 0 0 4 | 7 0 5 
0 9 0 | 0 5 0 | 2 0 0 

Solution found:
7 8 4 | 9 3 2 | 1 5 6 
6 1 9 | 4 8 5 | 3 2 7 
2 3 5 | 1 7 6 | 4 8 9 
---------------------
5 7 8 | 2 6 1 | 9 3 4 
3 4 1 | 8 9 7 | 5 6 2 
9 2 6 | 5 4 3 | 8 7 1 
---------------------
4 5 3 | 7 2 9 | 6 1 8 
8 6 2 | 3 1 4 | 7 9 5 
1 9 7 | 6 5 8 | 2 4 3 

Backtrack function calls: 50
Backtrack failures: 0
Success rate: 100.0%


Solving: medium.txt
Initial board:
5 3 0 | 0 7 0 | 0 0 0 
6 0 0 | 1 9 5 | 0 0 0 
0 9 8 | 0 0 0 | 0 6 0 
---------------------
8 0 0 | 0 6 0 | 0 0 3 
4 0 0 | 8 0 3 | 0 0 1 
7 0 0 | 0 2 0 | 0 0 6 
---------------------
0 6 0 | 0 0 0 | 2 8 0 
0 0 0 | 4 1 9 | 0 0 5 
0 0 0 | 0 8 0 | 0 7 9 

Solution found:
5 3 4 | 6 7 8 | 9 1 2 
6 7 2 | 1 9 5 | 3 4 8 
1 9 8 | 3 4 2 | 5 6 7 
-

[[8, 1, 2, 7, 5, 3, 6, 4, 9],
 [9, 4, 3, 6, 8, 2, 1, 7, 5],
 [6, 7, 5, 4, 9, 1, 2, 8, 3],
 [1, 5, 4, 2, 3, 7, 8, 9, 6],
 [3, 6, 9, 8, 4, 5, 7, 2, 1],
 [2, 8, 7, 1, 6, 9, 5, 3, 4],
 [5, 2, 1, 9, 7, 4, 3, 6, 8],
 [4, 3, 8, 5, 2, 6, 9, 1, 7],
 [7, 9, 6, 3, 1, 8, 4, 5, 2]]